In [1]:
import pandas as pd
import numpy as np
import os
from IPython.display import display

# Ensure processed directory exists
os.makedirs('../data/processed', exist_ok=True)
print("Libraries imported and directories set up.")

Libraries imported and directories set up.


Phase 1: Data Setup & Cleaning
1.1 Load Data

In [2]:
# Load data, skipping row 1 (Ticker) and row 2 (Date,,,,)
df = pd.read_csv('../data/raw/nsei_daily.csv', skiprows=[1, 2])

# The first column is mislabeled as 'Price', rename it to 'Date'
df = df.rename(columns={'Price': 'Date'})

# Display the first 5 rows
display(df.head())

,Date,Close,High,Low,Open,Volume
0,2020-01-01,12182.500000,12222.200195,12165.299805,12202.150391,304100
1,2020-01-02,12282.200195,12289.900391,12195.250000,12198.549805,407700
2,2020-01-03,12226.650391,12265.599609,12191.349609,12261.099609,428800
3,2020-01-06,11993.049805,12179.099609,11974.200195,12170.599609,396500
4,2020-01-07,12052.950195,12152.150391,12005.349609,12079.099609,447800


1.2 Sort and Index by Date

In [3]:
# Convert to datetime and set as index
df['Date'] = pd.to_datetime(df['Date'])
df = df.set_index('Date')

# Sort chronologically
df = df.sort_index()

# Ensure all OHLCV columns are numeric
numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Display info and first few rows
display(df.info())
display(df.head())

<class 'pandas.DataFrame'>
DatetimeIndex: 1670 entries, 2020-01-01 to 2026-09-30
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Close   1670 non-null   float64
 1   High    1670 non-null   float64
 2   Low     1670 non-null   float64
 3   Open    1670 non-null   float64
 4   Volume  1670 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 78.3 KB


None

,Close,High,Low,Open,Volume
Date,,,,,
2020-01-01,12182.500000,12222.200195,12165.299805,12202.150391,304100
2020-01-02,12282.200195,12289.900391,12195.250000,12198.549805,407700
2020-01-03,12226.650391,12265.599609,12191.349609,12261.099609,428800
2020-01-06,11993.049805,12179.099609,11974.200195,12170.599609,396500
2020-01-07,12052.950195,12152.150391,12005.349609,12079.099609,447800


1.3 Check Duplicates & Missing Values

In [4]:
# Check duplicates
duplicates = df.index.duplicated().sum()
print(f"Number of duplicate dates: {duplicates}")
if duplicates > 0:
    df = df[~df.index.duplicated(keep='first')]
    print("Dropped duplicate dates.")

# Check missing values
missing = df.isnull().sum()
print("\nMissing values before handling:")
print(missing)

if missing.sum() > 0:
    df = df.ffill()
    print("\nMissing values after forward-fill:")
    print(df.isnull().sum())
else:
    print("\nNo missing values found.")

Number of duplicate dates: 0

Missing values before handling:
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

No missing values found.


1.4 Validate OHLC Logic

In [5]:
invalid_rows = df[
    (df['Low'] > df['Open']) | 
    (df['Low'] > df['Close']) | 
    (df['High'] < df['Open']) | 
    (df['High'] < df['Close']) |
    (df['High'] < df['Low'])
]

print(f"Number of rows violating OHLC logic: {len(invalid_rows)}")
if not invalid_rows.empty:
    display(invalid_rows.head())
    df = df.drop(invalid_rows.index)
    print("Removed invalid rows.")
else:
    print("All OHLC logic checks passed.")

Number of rows violating OHLC logic: 0
All OHLC logic checks passed.


1.5 Handle Outliers (Inspect)

In [8]:
display(df[['Return', 'High_Low_Range', 'Volume_Change']].describe())
print("Note: Extreme values in 'Return' likely represent genuine market shocks (e.g., 2020 crash) and should be kept for volatility modeling.")

c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,Return,High_Low_Range,Volume_Change
count,1667.000000,1667.000000,1667.000000
mean,0.000368,202.594860,inf
std,0.011283,117.806324,NaN
min,-0.139038,46.951172,-1.000000
25%,-0.004391,128.250000,-0.135491
50%,0.000665,175.799805,-0.005065
75%,0.006088,243.175781,0.154517
max,0.084003,1898.050781,inf


Note: Extreme values in 'Return' likely represent genuine market shocks (e.g., 2020 crash) and should be kept for volatility modeling.


1.6 Create Base Derived Columns

In [6]:
# Create log returns
df['Return'] = np.log(df['Close'] / df['Close'].shift(1))

# Create High-Low Range
df['High_Low_Range'] = df['High'] - df['Low']

# Create Volume Change
df['Volume_Change'] = df['Volume'].pct_change()

# Drop the first row (NaN from shift and pct_change)
df = df.dropna()

display(df.head())

,Close,High,Low,Open,Volume,Return,High_Low_Range,Volume_Change
Date,,,,,,,,
2020-01-02,12282.200195,12289.900391,12195.250000,12198.549805,407700,0.008151,94.650391,0.340677
2020-01-03,12226.650391,12265.599609,12191.349609,12261.099609,428800,-0.004533,74.250000,0.051754
2020-01-06,11993.049805,12179.099609,11974.200195,12170.599609,396500,-0.019291,204.899414,-0.075326
2020-01-07,12052.950195,12152.150391,12005.349609,12079.099609,447800,0.004982,146.800781,0.129382
2020-01-08,12025.349609,12044.950195,11929.599609,11939.099609,446000,-0.002293,115.350586,-0.004020


Save Cleaned Data

In [7]:
output_path = '../data/processed/nsei_cleaned.csv'
df.to_csv(output_path)
print(f"Phase 1 Complete! Cleaned data saved to {output_path}")
print(f"Final Dataset Shape: {df.shape}")

Phase 1 Complete! Cleaned data saved to ../data/processed/nsei_cleaned.csv
Final Dataset Shape: (1667, 8)
